# CUBIC MARS — Chicago / CTA-Ventra — **PS3: Root-Cause + Severity** (SageMaker)

**Two heads in one delivery** (per the standing v7.0 rules — PS3 is gated on **macro-F1**, never raw accuracy):

| Head | Target | Honest baseline | Gate |
|---|---|---|---|
| **Severity** | `failure_level_label` (Purchase Card / Product / All-Purchase / All-Functions / Bus-Reader-Assembly) | macro-F1 ≈ 0.74 (structured features) | macro-F1 ≥ 0.55 (provisional) |
| **Root-cause (component)** | `derived_component_type` (CSC_READER / PRINTER / BHU / CHU / GATE_MECH / COMMS) | macro-F1 ≈ 0.54 | macro-F1 ≥ 0.45 (provisional) |

**Honest scope banner.** The *true 9-class ServiceNow* root cause stays **BLOCKED** — `SVN_STAGE U_FS_FAULT_CODES / U_FS_ACTION_CODES = 0 rows` (feasibility ~70%). What we ship instead is a **derived component/subsystem root cause** built from the fault taxonomy + pre-incident behaviour — the honest, data-backed version. When Cubic activates the SVN_STAGE ETL, the 9-class model drops into the same `root_cause` head.

**Leakage discipline (this is where PS3 goes wrong).** The incident free-text *names the answer*, so `AE_FAULT_DESCRIPTION / AE_SYMPTOM / AE_PROBLEM / AE_RESOLUTION` and the 7 text-derived `desc_*_flag` columns are **hard-excluded** — training on them produced a fake 0.999 F1. Both heads train on **structured pre-incident device behaviour** (24h/7d event windows, subsystem counts, OOS history, component age). `sn_event_code_id` (the ServiceNow fault code, solo-AUC 0.758) is kept but **flagged as the dominant driver** and put through the solo-AUC ≤ 0.95 leakage gate.

**Device scope.** `TVM` (dominant), `GATE` (sparse — honest caveat), `VALIDATOR` = **0 availability-event incidents** by construction → honest stub + a device_event **OOS-proxy** (validators surface as OOS `Set` events, not availability events).

**Outputs (exhaustive — a superset of the prior `ps3_v3_outputs`).** Per device type and **per head**: model bake-off leaderboard; **per-model** confusion matrix + ROC(OVR) + PR(OVR) + feature-importance (LogReg/RF/HistGBM/XGBoost/LightGBM/CatBoost + an **Optuna-tuned XGBoost** challenger); champion confusion + per-class F1 + calibration/reliability; class-distribution; a full `classification_report` → `report.txt`; the SHAP suite (global importance + beeswarm + per-class mean-|SHAP| + dependence plots); a rich `champion_summary.json` (macro-F1, weighted-F1, accuracy, balanced-accuracy, ROC-AUC, **PR-AUC**, Cohen-κ, MCC, log-loss, per-class precision/recall/F1/support); plus the RDS-ready incident/device/**serial** prediction feeds and the joblib serving bundle. Every cell tees to `run_console_log.txt`; cross-device figures + RDS load manifest + model-registry rows land at the `PS3_outputs_pavan/` root. Each figure renders inline **and** is saved.

### Dependencies (SageMaker Studio kernel)
```bash
%pip install -q "numpy<2" pandas scikit-learn lightgbm xgboost catboost shap optuna pyarrow s3fs joblib matplotlib
```
Run `notebooks/export_gold_to_s3.py` + `export_silver_to_s3.py` in Databricks **first** so the S3 parquet exports exist (SageMaker Studio has no Spark — it reads the S3 export, not Unity Catalog directly). On a Databricks cluster the loader auto-detects Spark and reads UC with zero config.

## 1. Configuration — the only block you edit
`SMOKE_TEST_SYNTHETIC=True` dry-runs the whole notebook offline on synthetic data (no S3), to prove it runs before you point it at the real export.

In [ ]:
# ----------------------------------------------------------------------------- CONFIG
CONFIG = {
    "CITY_ID": "CHI",
    "DEVICE_SCOPE": ["TVM", "GATE", "VALIDATOR"],   # engine loops these; VALIDATOR -> stub+proxy
    "SEVERITY_TARGET": "failure_level_label",         # multi-class severity head target
    "SEVERITY_COLLAPSE": {                             # -> RDS predicted_label (dashboard MAJOR/CRITICAL)
        # 2026-07-26 FIX: this map MUST be keyed on failure_level_label CODES, not on the
        # human-readable labels. pred_severity holds codes ("ALL_FUNCTIONS"), so the previous
        # label-keyed map missed on every row and .fillna("MAJOR") silently marked 100% of
        # incidents MAJOR -- which drove pct_critical_pred to 0.0 fleet-wide in the 19-Jul run.
        "PURCHASE_CARD": "MAJOR", "PURCHASE_PRODUCT": "MAJOR", "NONPAYMENT": "MAJOR",
        "ALL_PURCHASE": "CRITICAL", "ALL_FUNCTIONS": "CRITICAL",
        "BUS_READER": "CRITICAL", "BUS_READER_ASSEMBLY": "CRITICAL",
    },
    "ROOTCAUSE_TARGET": "derived_component_type",      # component/subsystem root-cause head target
    "SEVERITY_F1_FLOOR": 0.55,                         # provisional macro-F1 gate (honest v3 = 0.74)
    "ROOTCAUSE_F1_FLOOR": 0.45,                        # provisional macro-F1 gate (honest v3 = 0.54)
    "MIN_CLASS_COUNT": 30,                             # collapse classes rarer than this into OTHER
    "MIN_INCIDENTS_TO_MODEL": 200,                     # below this -> report-only (sparse device type)

    # ---- data source (SageMaker Studio path = S3 parquet; Spark auto-detected on Databricks) ----
    "DATA_SOURCE": "auto",
    "GOLD_TABLE":   "mars_dev.gold.device_ps3_incident",
    "GOLD_PARQUET": "s3://cubic-mars-pm-s3-datalake-dev-gold-170202974600/chicago/gold/device_ps3_incident",
    "S3_REGION":    "us-east-1",
    "DBX_HOST": "", "DBX_HTTP_PATH": "", "DBX_TOKEN_ENV": "DATABRICKS_TOKEN",

    # ---- enrichment silvers (leakage-safe, prior-window; each guarded -> skip if export missing) ----
    "ENRICH_SOURCES": True,
    "SILVER_PREFIX": "s3://cubic-mars-pm-s3-datalake-dev-gold-170202974600/chicago/silver",
    "ENRICH_USAGE":     True,   # usage_lifecycle_daily -- failure_count_30d, days_since_last_*, wear
    "ENRICH_STATION":   True,   # station_network_daily -- prior-day station co-failure stress
    "HWCONFIG_TABLE":   "mars_dev.silver.hw_config_current",

    # ---- feature policy (PS3 content-leakage is the #1 risk: free text NAMES the answer) ----
    # HARD leak list -- excluded from BOTH heads (fault text + text-derived flags + targets).
    "EXPLICIT_LEAK": [
        "AE_FAULT_DESCRIPTION", "AE_SYMPTOM", "AE_PROBLEM", "AE_RESOLUTION", "combined_text",
        "affected_component", "matched_component", "sn_event_code_name",
        "desc_printer_flag", "desc_card_reader_flag", "desc_bill_handler_flag", "desc_coin_flag",
        "desc_comms_flag", "desc_timeout_flag", "desc_replacement_flag",
        "failure_level", "failure_level_label", "is_chargeable", "is_device_fault",
        "derived_component_type", "root_cause_category", "kpi_rule_id", "kpi_category_name",
        "AE_FAULT_STATE", "wot_state", "request_type",
    ],
    # CONCURRENT / post-incident cols -- known only after the incident; ablation only, not champion.
    "CONCURRENT_FEATURES": ["incident_duration_min"],
    "USE_CONCURRENT_FEATURES": False,
    # MONITORED -- kept but flagged (the ServiceNow fault code dominates; leakage-scanned).
    "MONITORED_FEATURES": ["sn_event_code_id", "sn_priority"],
    "LEAKAGE_AUC_THRESHOLD": 0.95,
    "NZV_UNIQUE_MIN": 2,
    "RUN_OPTUNA": True,          # add an Optuna-tuned XGBoost challenger (matches v3's xgb_optuna)
    "N_OPTUNA_TRIALS": 20,
    "TUNE_SAMPLE_ROWS": 60000,   # cap tuning rows for speed (None = all)

    # ---- split / CV ----
    "TEST_FRACTION": 0.20, "EMBARGO_DAYS": 2, "CV_SPLITS": 4,
    "MIN_TRAIN_DATE": "2024-01-01", "RANDOM_STATE": 42,

    # ---- outputs ----
    "OUT_ROOT": "PS3_outputs_pavan",
    "SUBFOLDER": {"TVM": "tvm", "GATE": "gates", "VALIDATOR": "validators"},

    # ---- MLflow (guarded no-op if blank) ----
    "MLFLOW_TRACKING_URI": "", "MLFLOW_EXPERIMENT": "chicago-ps3-rootcause-severity", "AWS_REGION": "us-east-1",
}
RS = CONFIG["RANDOM_STATE"]
CONFIG["SMOKE_TEST_SYNTHETIC"] = False   # True -> offline synthetic dry-run (no S3 / no Spark)

## 2. Environment, graceful imports & the run-log tee
Optional boosters degrade gracefully. `start_run_log` tees **every** cell's stdout+stderr into `PS3_outputs_pavan/run_console_log.txt` (fresh per Run-All) so the whole run is captured for the dashboard/audit.

In [ ]:
%matplotlib inline
import os, sys, json, math, warnings, time
from pathlib import Path
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
# ----------------------------------------------------------------------------- imports (graceful)
def _try(name):
    try: return __import__(name)
    except Exception as e:
        print(f"[optional] {name} unavailable ({type(e).__name__}); paths using it will be skipped"); return None
xgb = _try("xgboost"); lgb = _try("lightgbm"); catb = _try("catboost"); shap_mod = _try("shap")
optuna_mod = _try("optuna")
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, LabelEncoder, label_binarize
from sklearn.metrics import (f1_score, roc_auc_score, accuracy_score, confusion_matrix,
                             classification_report, precision_score, recall_score,
                             average_precision_score, balanced_accuracy_score, cohen_kappa_score,
                             matthews_corrcoef, precision_recall_curve, roc_curve, log_loss)
np.random.seed(RS)

PASTEL = ["#9DC3E6", "#A9D18E", "#F4B183", "#C9A0DC", "#FFD966", "#F28E8E", "#8FD4C8", "#BFBFBF"]

# ----------------------------------------------------------------------------- tee logger
class _Tee:
    def __init__(self, *streams): self.streams = streams
    def write(self, d):
        for s in self.streams:
            try: s.write(d); s.flush()
            except Exception: pass
    def flush(self):
        for s in self.streams:
            try: s.flush()
            except Exception: pass

def start_run_log(out_root):
    Path(out_root).mkdir(parents=True, exist_ok=True)
    if "_ORIG_STDOUT" not in globals():
        globals()["_ORIG_STDOUT"], globals()["_ORIG_STDERR"] = sys.stdout, sys.stderr
    lg = open(Path(out_root) / "run_console_log.txt", "w", encoding="utf-8")
    sys.stdout = _Tee(globals()["_ORIG_STDOUT"], lg); sys.stderr = _Tee(globals()["_ORIG_STDERR"], lg)
    print(f"[log] capturing all cell output to {Path(out_root)/'run_console_log.txt'}")

start_run_log(CONFIG["OUT_ROOT"])

## 3. Data loader (Spark → databricks-sql → S3 parquet) + synthetic generator
Same robust loader as the PS1 engine: active Spark session first (Databricks), then databricks-sql, then the S3 parquet export (the SageMaker Studio default). DECIMAL→float coercion avoids the `Decimal/float` TypeError. The synthetic generator is only used when `SMOKE_TEST_SYNTHETIC=True`.

In [ ]:
# ----------------------------------------------------------------------------- loader (Spark->SQL->parquet)
def _active_spark():
    g = globals().get("spark", None)
    if g is not None: return g
    try:
        from pyspark.sql import SparkSession
        return SparkSession.getActiveSession()
    except Exception: return None

def load_table(table, parquet_path, category_filter=None, required=True):
    src = CONFIG["DATA_SOURCE"]; sp = _active_spark()
    try:
        if src in ("auto", "databricks_spark") and sp is not None:
            sdf = sp.table(table)
            if category_filter: sdf = sdf.filter(f"mars_device_category = '{category_filter}'")
            print(f"[load] Spark -> {table}"); return sdf.toPandas()
        if src in ("auto", "parquet") and parquet_path:
            rk = {}
            if str(parquet_path).startswith("s3"):
                rk["storage_options"] = {"client_kwargs": {"region_name": CONFIG.get("S3_REGION", "us-east-1")}}
            print(f"[load] parquet -> {parquet_path}")
            if category_filter:
                try: return pd.read_parquet(parquet_path, filters=[("mars_device_category","==",category_filter)], **rk)
                except Exception: pass
            return pd.read_parquet(parquet_path, **rk)
    except Exception as e:
        if required: raise
        print(f"[load] optional source {table} unavailable ({type(e).__name__}: {str(e)[:80]}); skipping"); return None
    if required:
        raise RuntimeError(f"No usable data source for {table}. On SageMaker Studio run "
                           "export_gold_to_s3.py + export_silver_to_s3.py first, or run on a Databricks cluster.")
    return None

def coerce_decimals(df):
    # parquet returns Spark DECIMAL as python Decimal (object dtype) -> float (else TypeErrors downstream)
    from decimal import Decimal
    for c in df.columns:
        if df[c].dtype == object:
            s = df[c].dropna()
            if len(s) and isinstance(s.iloc[0], Decimal):
                df[c] = pd.to_numeric(df[c], errors="coerce")
    return df

# ----------------------------------------------------------------------------- synthetic generator (SMOKE TEST ONLY)
def make_synthetic(n_tvm=6000, n_gate=350, seed=7):
    rng = np.random.default_rng(seed)
    comp_types = ["CSC_READER", "PRINTER", "BHU", "CHU", "GATE_MECH", "COMMS"]
    sev_levels = {2: "Purchase Card Functions", 3: "Purchase Product Functions",
                  4: "All Purchase Functions", 5: "All Functions", 16: "Bus Reader Assembly Fail"}
    def gen(cat, n, dev_prefix, comp_probs):
        comp = rng.choice(comp_types, n, p=comp_probs)
        # subsystem event counts correlated with the true component (honest pre-incident signal)
        base = {c: rng.poisson(0.4, n).astype(float) for c in
                ["bhu", "chu", "printer", "gate_mech", "csc_reader", "comms"]}
        keymap = {"CSC_READER": "csc_reader", "PRINTER": "printer", "BHU": "bhu", "CHU": "chu",
                  "GATE_MECH": "gate_mech", "COMMS": "comms"}
        for i, c in enumerate(comp):
            base[keymap[c]][i] += rng.poisson(4)
        # severity partly driven by component + age + event load (honest, imperfect)
        age = rng.integers(20, 3000, n).astype(float)
        load = sum(base.values())
        sev_score = (0.5 * (comp == "GATE_MECH") + 0.4 * (comp == "COMMS") + 0.0006 * age
                     + 0.05 * load + rng.normal(0, 0.6, n))
        lvl = np.where(sev_score > 1.6, 16, np.where(sev_score > 1.1, 5,
              np.where(sev_score > 0.7, 4, np.where(sev_score > 0.3, 3, 2))))
        sn_code = (lvl * 100 + rng.integers(0, 40, n)).astype(int)   # fault code ~ severity (dominant, monitored)
        dt = pd.to_datetime("2024-01-01") + pd.to_timedelta(rng.integers(0, 800, n), unit="D") \
             + pd.to_timedelta(rng.integers(0, 24 * 3600, n), unit="s")
        dev_id = [f"{dev_prefix}{rng.integers(1, max(2, n // 6)):05d}" for _ in range(n)]
        return pd.DataFrame({
            "availability_event_id": [f"AE-{cat}-{i:07d}" for i in range(n)],
            "device_id": dev_id, "mars_device_category": cat,
            "AE_START_DTM": dt, "AE_END_DTM": dt + pd.to_timedelta(rng.integers(5, 300, n), unit="m"),
            "transit_day": dt.normalize(),
            "incident_duration_min": rng.integers(5, 300, n).astype(float),
            "failure_level": lvl, "failure_level_label": [sev_levels[int(x)] for x in lvl],
            "derived_component_type": comp, "root_cause_category": comp,
            "affected_component": comp, "matched_component": comp,
            "AE_FAULT_DESCRIPTION": [f"{c} fault detected" for c in comp],   # leak col (must be dropped)
            "AE_SYMPTOM": ["symptom"] * n, "AE_PROBLEM": ["problem"] * n, "AE_RESOLUTION": ["replaced part"] * n,
            "desc_card_reader_flag": (comp == "CSC_READER").astype(int),      # text-derived leak flags
            "desc_printer_flag": (comp == "PRINTER").astype(int),
            "desc_bill_handler_flag": (comp == "BHU").astype(int),
            "desc_coin_flag": (comp == "CHU").astype(int),
            "desc_comms_flag": (comp == "COMMS").astype(int),
            "desc_timeout_flag": rng.integers(0, 2, n), "desc_replacement_flag": rng.integers(0, 2, n),
            "events_24h_prior": load + rng.poisson(1, n),
            "oos_onsets_24h": rng.poisson(0.6, n).astype(float),
            "bhu_events_24h": base["bhu"], "chu_events_24h": base["chu"], "printer_events_24h": base["printer"],
            "gate_mech_events_24h": base["gate_mech"], "csc_reader_events_24h": base["csc_reader"],
            "comms_events_24h": base["comms"],
            "oos_onsets_7d_prior": rng.poisson(2, n).astype(float), "events_7d_prior": load * 3 + rng.poisson(5, n),
            "component_age_days": age,
            "matched_serial_nbr": [f"SN{cat}{rng.integers(1, max(2, n // 3)):06d}" for _ in range(n)],
            "sn_event_code_id": sn_code, "sn_event_code_name": [f"code_{x}" for x in sn_code],
            "sn_priority": rng.integers(1, 5, n).astype(float),
            "kpi_rule_id": [None] * n, "kpi_category_name": [None] * n,
            "FACILITY_ID": [f"FAC-{rng.integers(1, 60):04d}" for _ in range(n)],
            "FACILITY_NAME": ["Station"] * n, "OPERATOR_ID": [1] * n, "OPERATOR_NAME": ["CTA"] * n,
            "DEVICE_NAME": dev_id, "DEVICE_TYPE_NAME": cat, "DEVICE_SERIAL_NUMBER": dev_id,
            "is_chargeable": [True] * n, "is_device_fault": [True] * n,
            "wot_state": ["Closed"] * n, "request_type": ["Corrective"] * n, "AE_FAULT_STATE": ["Set"] * n,
        })
    tvm = gen("TVM", n_tvm, "TVM", [0.55, 0.18, 0.12, 0.08, 0.0, 0.07])
    gate = gen("GATE", n_gate, "RVG", [0.15, 0.0, 0.0, 0.0, 0.6, 0.25])
    return pd.concat([tvm, gate], ignore_index=True)

def make_synthetic_validator_failures(n=1500, seed=11):
    rng = np.random.default_rng(seed)
    dt = pd.to_datetime("2024-01-01") + pd.to_timedelta(rng.integers(0, 800, n), unit="D")
    return pd.DataFrame({"DEVICE_ID": [f"BMV{rng.integers(1,300):05d}" for _ in range(n)],
                         "mars_device_category": "VALIDATOR", "ledger_date": dt.normalize(),
                         "failure_level": rng.choice([2, 4, 5, 16], n, p=[0.5, 0.3, 0.1, 0.1]),
                         "fault_description": ["OOS"] * n})

_orig_load = load_table
def load_table(table, parquet_path, category_filter=None, required=True):   # noqa: F811
    if "device_failures" in str(table) and "_SYNTH_VALIDATOR_FAILS" in globals():
        d = globals()["_SYNTH_VALIDATOR_FAILS"]
        return d[d["mars_device_category"] == category_filter].copy() if category_filter else d
    return _orig_load(table, parquet_path, category_filter, required)

## 4. Load `gold.device_ps3_incident`
One row per `availability_event_id` (TVM + GATE; VALIDATOR = 0). 62 columns — targets, pre-incident event windows, `matched_serial_nbr` + `component_age_days` (the serial link), SN context.

In [ ]:
if CONFIG.get("SMOKE_TEST_SYNTHETIC"):
    print("[MODE] SYNTHETIC dry-run -- no S3/Spark used")
    df = make_synthetic()
    globals()["_SYNTH_VALIDATOR_FAILS"] = make_synthetic_validator_failures()
else:
    df = load_table(CONFIG["GOLD_TABLE"], CONFIG["GOLD_PARQUET"])
    df = coerce_decimals(df)
if "transit_day" in df.columns:
    df = df[df["transit_day"].astype(str) >= CONFIG["MIN_TRAIN_DATE"]].copy()
print(f"loaded {len(df):,} PS3 incidents | by type: {df['mars_device_category'].value_counts().to_dict()}")

## 5. Cross-device overview EDA (rich visualization)
Severity distribution by device type, derived-component distribution, incident volume over time, component-age-by-severity, and the **pre-incident subsystem-intensity → component** heatmap (the honest signal the root-cause head learns). Figures render inline and save to the `PS3_outputs_pavan/` root.

In [ ]:
# ----------------------------------------------------------------------------- feature engineering
def add_temporal(df):
    dt = pd.to_datetime(df["AE_START_DTM"], errors="coerce")
    df["ae_hour"] = dt.dt.hour.fillna(12).astype("float64")
    df["ae_dow"] = dt.dt.dayofweek.fillna(0).astype("float64")
    df["ae_month"] = dt.dt.month.fillna(1).astype("float64")
    return df

def add_facility_freq(df):
    if "FACILITY_ID" in df.columns:
        fc = df["FACILITY_ID"].astype(str).map(df["FACILITY_ID"].astype(str).value_counts())
        df["facility_incident_freq"] = fc.fillna(0).astype("float64")
    return df

BASE_STRUCT = ["events_24h_prior", "oos_onsets_24h", "bhu_events_24h", "chu_events_24h",
               "printer_events_24h", "gate_mech_events_24h", "csc_reader_events_24h",
               "comms_events_24h", "oos_onsets_7d_prior", "events_7d_prior", "component_age_days",
               "ae_hour", "ae_dow", "ae_month", "facility_incident_freq"]

def build_feature_list(df):
    feats = [c for c in BASE_STRUCT if c in df.columns]
    for m in CONFIG["MONITORED_FEATURES"]:
        if m in df.columns: feats.append(m)
    if CONFIG["USE_CONCURRENT_FEATURES"]:
        feats += [c for c in CONFIG["CONCURRENT_FEATURES"] if c in df.columns]
    # enrichment cols added by joins (prefix enr_)
    feats += [c for c in df.columns if c.startswith("enr_")]
    # never allow a leak/target column in features
    feats = [c for c in feats if c not in set(CONFIG["EXPLICIT_LEAK"])]
    return list(dict.fromkeys(feats))    # de-dup, keep order

# ----------------------------------------------------------------------------- overview EDA (root folder)
def overview_eda(df, out_root):
    out = Path(out_root)
    print("\n" + "=" * 74); print(" CROSS-DEVICE OVERVIEW (device_ps3_incident)"); print("=" * 74)
    print(f" incidents {len(df):,} | devices {df['device_id'].nunique():,} | "
          f"{pd.to_datetime(df['AE_START_DTM']).min().date()} -> {pd.to_datetime(df['AE_START_DTM']).max().date()}")
    print(" by device type:"); print(df["mars_device_category"].value_counts().to_string())
    # severity distribution by device type
    if CONFIG["SEVERITY_TARGET"] in df.columns:
        ct = pd.crosstab(df["mars_device_category"], df[CONFIG["SEVERITY_TARGET"]].astype(str))
        ax = ct.plot(kind="bar", stacked=True, figsize=(9, 5), color=PASTEL[:ct.shape[1]])
        ax.set_title("PS3 -- severity distribution by device type"); ax.legend(fontsize=7, ncol=2)
        plt.tight_layout(); plt.savefig(out / "ps3_overview_severity_by_type.png", dpi=110); plt.show()
    # component distribution
    if CONFIG["ROOTCAUSE_TARGET"] in df.columns:
        vc = df[CONFIG["ROOTCAUSE_TARGET"]].astype(str).value_counts().head(12)
        ax = vc[::-1].plot(kind="barh", figsize=(8, 5), color="#A9D18E")
        ax.set_title("PS3 -- root-cause component distribution (derived)"); plt.tight_layout()
        plt.savefig(out / "ps3_overview_component_distribution.png", dpi=110); plt.show()
    # incidents over time by severity
    tmp = df.copy(); tmp["month"] = pd.to_datetime(tmp["AE_START_DTM"]).dt.to_period("M").astype(str)
    piv = tmp.groupby("month").size()
    fig, ax = plt.subplots(figsize=(10, 4)); piv.plot(ax=ax, color="#9DC3E6", marker="o", ms=3)
    ax.set_title("PS3 -- incident volume over time"); ax.set_xlabel(""); plt.xticks(rotation=45, fontsize=7)
    plt.tight_layout(); plt.savefig(out / "ps3_overview_incidents_over_time.png", dpi=110); plt.show()
    # component age by severity (box)
    if {"component_age_days", CONFIG["SEVERITY_TARGET"]}.issubset(df.columns):
        d2 = df.dropna(subset=["component_age_days"])
        cats = d2[CONFIG["SEVERITY_TARGET"]].astype(str).value_counts().head(6).index.tolist()
        data = [d2[d2[CONFIG["SEVERITY_TARGET"]].astype(str) == c]["component_age_days"].values for c in cats]
        if any(len(x) for x in data):
            fig, ax = plt.subplots(figsize=(9, 5)); ax.boxplot(data, labels=cats, showfliers=False)
            ax.set_title("PS3 -- component age by severity"); ax.set_ylabel("component age (days)")
            plt.xticks(rotation=30, ha="right", fontsize=8); plt.tight_layout()
            plt.savefig(out / "ps3_overview_age_by_severity.png", dpi=110); plt.show()
    # subsystem pre-incident intensity by component (honest signal heatmap)
    subs = [c for c in ["bhu_events_24h","chu_events_24h","printer_events_24h","gate_mech_events_24h",
                        "csc_reader_events_24h","comms_events_24h"] if c in df.columns]
    if subs and CONFIG["ROOTCAUSE_TARGET"] in df.columns:
        comp = df[CONFIG["ROOTCAUSE_TARGET"]].astype(str)
        keep = comp.value_counts().head(7).index
        M = df[comp.isin(keep)].groupby(comp)[subs].mean().loc[[c for c in keep if c in comp.unique()]]
        fig, ax = plt.subplots(figsize=(8, 5)); im = ax.imshow(M.values, cmap="YlOrBr", aspect="auto")
        ax.set_xticks(range(len(subs))); ax.set_xticklabels([s.replace("_events_24h","") for s in subs], rotation=40, ha="right", fontsize=8)
        ax.set_yticks(range(len(M))); ax.set_yticklabels(M.index, fontsize=8)
        ax.set_title("PS3 -- mean pre-incident subsystem events (24h) by component"); fig.colorbar(im, fraction=0.046)
        plt.tight_layout(); plt.savefig(out / "ps3_overview_subsystem_by_component.png", dpi=110); plt.show()
    print(" overview figures written to", out.resolve())

overview_eda(df, CONFIG["OUT_ROOT"])

## 6. The PS3 engine (both heads)

One reusable engine, applied per device type:

1. **Leakage scan** — solo-AUC one-vs-rest on the severity majority class; anything > 0.95 is dropped. The hard `EXPLICIT_LEAK` list (fault text + `desc_*_flag` + targets) is removed up front — the scan is the backstop, the exclusion list is the real guard. *(We scan on the severity target, not the component target: a pre-incident subsystem count that strongly predicts its own component is legitimate signal, not leakage, and a solo-AUC scan on the component target would wrongly nuke it.)*
2. **Purged temporal split** — train on the earliest 80% of incident dates, embargo `EMBARGO_DAYS`, test on the most-recent tail (realistic generalization; no future rows in train).
3. **Multi-class bake-off** — LogReg / RandomForest / HistGBM / XGBoost / LightGBM (+ CatBoost if present), class-balanced. Champion = best **macro-F1**.
4. **Gate** — macro-F1 vs the provisional floor; PASS/FAIL reported honestly (never gated on accuracy — rare-severity base rates would make accuracy a lie).
5. **Explainability** — TreeSHAP global drivers + per-class F1 + confusion matrix per head.
6. **Scoring** — incident-grain predictions for both heads, rolled up to **device grain** and **serial grain** (`matched_serial_nbr`) — the device-id + serial-id feeds the dashboard/RDS read.

In [ ]:
# ----------------------------------------------------------------------------- leakage scan
def leakage_scan(df, target_col, feats, split_date):
    tr = df[df["_ae_date"] <= split_date]; va = df[df["_ae_date"] > split_date]
    if va[target_col].nunique() < 2 or len(va) < 20:
        print("[leak] holdout too small for solo-AUC scan; skipping"); return pd.Series(dtype=float), []
    top = va[target_col].value_counts().index[0]
    ybin = (va[target_col] == top).astype(int)
    def solo(col):
        a = pd.to_numeric(tr[col], errors="coerce"); b = pd.to_numeric(va[col], errors="coerce")
        med = a.median()
        a = a.fillna(med); b = b.fillna(med)
        if a.nunique() < 2: return np.nan
        try:
            auc = roc_auc_score(ybin, b); return max(auc, 1 - auc)
        except Exception: return np.nan
    scan = pd.Series({c: solo(c) for c in feats}).dropna().sort_values(ascending=False)
    leaky = scan[scan > CONFIG["LEAKAGE_AUC_THRESHOLD"]].index.tolist()
    print("top solo-AUC features (one-vs-rest on majority class):")
    print(scan.head(10).round(3).to_string())
    print(f"leakage gate {CONFIG['LEAKAGE_AUC_THRESHOLD']} -> {'CLEAN' if not leaky else 'DROP '+str(leaky)}")
    return scan, leaky

# ----------------------------------------------------------------------------- target prep
def prep_target(df, target_col):
    y = df[target_col].astype(str).fillna("UNKNOWN")
    vc = y.value_counts()
    rare = vc[vc < CONFIG["MIN_CLASS_COUNT"]].index.tolist()
    if rare:
        y = y.where(~y.isin(rare), "OTHER")
        print(f"[target:{target_col}] collapsed {len(rare)} rare class(es) (<{CONFIG['MIN_CLASS_COUNT']}) -> OTHER: {rare}")
    return y

# ----------------------------------------------------------------------------- model factory
def make_prep(feats):
    return ColumnTransformer([("num", Pipeline([("imp", SimpleImputer(strategy="median")),
                                                ("sc", StandardScaler())]), feats)], remainder="drop")

def builders(n_classes, spw=None):
    b = {}
    b["LogReg"] = lambda: LogisticRegression(max_iter=2000, class_weight="balanced")
    b["RandomForest"] = lambda: RandomForestClassifier(n_estimators=300, class_weight="balanced_subsample",
                                                       n_jobs=-1, random_state=RS)
    b["HistGBM"] = lambda: HistGradientBoostingClassifier(max_iter=300, learning_rate=0.06, random_state=RS)
    if xgb is not None:
        b["XGBoost"] = lambda: xgb.XGBClassifier(
            n_estimators=350, max_depth=6, learning_rate=0.06, subsample=0.85, colsample_bytree=0.8,
            objective="multi:softprob", num_class=n_classes, eval_metric="mlogloss",
            tree_method="hist", random_state=RS, n_jobs=-1)
    if lgb is not None:
        b["LightGBM"] = lambda: lgb.LGBMClassifier(
            n_estimators=400, num_leaves=48, learning_rate=0.05, subsample=0.85, colsample_bytree=0.8,
            objective="multiclass", num_class=n_classes, class_weight="balanced", random_state=RS, n_jobs=-1, verbose=-1)
    if catb is not None:
        b["CatBoost"] = lambda: catb.CatBoostClassifier(iterations=400, depth=6, learning_rate=0.06,
                                                        loss_function="MultiClass", verbose=0, random_seed=RS)
    return b

def sample_weight(y):
    vc = pd.Series(y).value_counts()
    w = {k: len(y) / (len(vc) * v) for k, v in vc.items()}
    return pd.Series(y).map(w).values

def fit_one(model, prep, X, y):
    Xt = prep.transform(X)
    try:
        model.fit(Xt, y, sample_weight=sample_weight(y))
    except TypeError:
        model.fit(Xt, y)
    return model

def optuna_challenger(prep, Xtr, ytr, n_cls):
    """Optuna-tuned XGBoost challenger (matches v3's xgb_optuna). Honest temporal tune-split of TRAIN."""
    if optuna_mod is None or xgb is None:
        return None
    Xt = prep.transform(Xtr); y = np.asarray(ytr)
    cap = CONFIG.get("TUNE_SAMPLE_ROWS")
    if cap and len(Xt) > cap:
        Xt, y = Xt[-cap:], y[-cap:]                       # most-recent rows (temporal)
    cut = int(len(Xt) * 0.8)
    Xa, Xb, ya, yb = Xt[:cut], Xt[cut:], y[:cut], y[cut:]
    if len(np.unique(ya)) < 2 or len(np.unique(yb)) < 2:
        return None
    optuna_mod.logging.set_verbosity(optuna_mod.logging.WARNING)

    def objective(t):
        p = dict(n_estimators=t.suggest_int("n_estimators", 200, 500, step=100),
                 max_depth=t.suggest_int("max_depth", 3, 9),
                 learning_rate=t.suggest_float("learning_rate", 0.02, 0.2, log=True),
                 subsample=t.suggest_float("subsample", 0.6, 1.0),
                 colsample_bytree=t.suggest_float("colsample_bytree", 0.6, 1.0))
        m = xgb.XGBClassifier(objective="multi:softprob", num_class=n_cls, eval_metric="mlogloss",
                              tree_method="hist", random_state=RS, n_jobs=-1, **p)
        m.fit(Xa, ya)
        return f1_score(yb, m.predict(Xb), average="macro", zero_division=0)

    study = optuna_mod.create_study(direction="maximize",
                                    sampler=optuna_mod.samplers.TPESampler(seed=RS))
    study.optimize(objective, n_trials=CONFIG["N_OPTUNA_TRIALS"], show_progress_bar=False)
    best = xgb.XGBClassifier(objective="multi:softprob", num_class=n_cls, eval_metric="mlogloss",
                             tree_method="hist", random_state=RS, n_jobs=-1, **study.best_params)
    best.fit(Xt, y)                                       # refit on the full (capped) train
    return best, float(study.best_value), study.best_params

# ----------------------------------------------------------------------------- head trainer
def train_head(df, feats, target_col, head_name, cat, out, split_date):
    print(f"\n----- {head_name} head ({target_col}) -----")
    y_all = prep_target(df, target_col)
    classes = sorted(y_all.unique().tolist())
    n_cls = len(classes)
    if n_cls < 2:
        print(f"[{head_name}] only {n_cls} class present -> cannot model; skipping head"); return None
    le = LabelEncoder().fit(y_all)
    df = df.copy(); df["_y"] = le.transform(y_all)

    tr = df[df["_ae_date"] <= split_date].copy()
    embargo = split_date + pd.Timedelta(days=CONFIG["EMBARGO_DAYS"])
    te = df[df["_ae_date"] > embargo].copy()
    if len(te) < 20 or len(tr) < 50:
        print(f"[{head_name}] split too small (train {len(tr)}, test {len(te)}) -> skipping head"); return None
    Xtr, ytr = tr[feats].astype("float64"), tr["_y"].values
    Xte, yte = te[feats].astype("float64"), te["_y"].values
    prep = make_prep(feats).fit(Xtr)

    LBL = np.arange(n_cls)                       # full class index set -> every metric aligns to le.classes_
    Yte_oh = label_binarize(yte, classes=LBL)    # one-vs-rest one-hot for AUC / PR-AUC
    if n_cls == 2 and Yte_oh.shape[1] == 1:
        Yte_oh = np.hstack([1 - Yte_oh, Yte_oh])
    rows = []; fitted = {}; proba_of = {}
    for name, mk in builders(n_cls).items():
        try:
            t0 = time.time(); m = fit_one(mk(), prep, Xtr, ytr)
            proba = m.predict_proba(prep.transform(Xte))
            pred = proba.argmax(axis=1)
            f1m = f1_score(yte, pred, labels=LBL, average="macro", zero_division=0)
            f1w = f1_score(yte, pred, labels=LBL, average="weighted", zero_division=0)
            acc = accuracy_score(yte, pred)
            try:
                auc = (roc_auc_score(yte, proba, average="macro", multi_class="ovr", labels=LBL)
                       if n_cls > 2 else roc_auc_score(yte, proba[:, 1]))
            except Exception: auc = np.nan
            try: prauc = average_precision_score(Yte_oh, proba, average="macro")
            except Exception: prauc = np.nan
            rows.append({"model": name, "f1_macro": f1m, "f1_weighted": f1w, "accuracy": acc,
                         "auc_macro_ovr": auc, "pr_auc_macro": prauc, "fit_s": round(time.time() - t0, 1)})
            fitted[name] = m; proba_of[name] = proba
            print(f"  {name:13s} F1-macro {f1m:.4f} | F1-w {f1w:.4f} | acc {acc:.4f} | "
                  f"AUC {auc if not np.isnan(auc) else float('nan'):.4f} | PR-AUC {prauc if not np.isnan(prauc) else float('nan'):.4f}")
        except Exception as e:
            print(f"  {name:13s} FAILED: {type(e).__name__}: {str(e)[:70]}")
    # Optuna-tuned XGBoost challenger (v3 parity: xgb_optuna) -> flows through all downstream artifacts
    if CONFIG.get("RUN_OPTUNA"):
        try:
            res = optuna_challenger(prep, Xtr, ytr, n_cls)
            if res is not None:
                best, bval, bparams = res
                proba = best.predict_proba(prep.transform(Xte)); pred = proba.argmax(1)
                try: auc = (roc_auc_score(yte, proba, average="macro", multi_class="ovr", labels=LBL)
                            if n_cls > 2 else roc_auc_score(yte, proba[:, 1]))
                except Exception: auc = np.nan
                try: prauc = average_precision_score(Yte_oh, proba, average="macro")
                except Exception: prauc = np.nan
                rows.append({"model": "XGBoost_optuna",
                             "f1_macro": f1_score(yte, pred, labels=LBL, average="macro", zero_division=0),
                             "f1_weighted": f1_score(yte, pred, labels=LBL, average="weighted", zero_division=0),
                             "accuracy": accuracy_score(yte, pred), "auc_macro_ovr": auc,
                             "pr_auc_macro": prauc, "fit_s": np.nan})
                fitted["XGBoost_optuna"] = best; proba_of["XGBoost_optuna"] = proba
                json.dump(bparams, open(out / f"{cat.lower()}_{head_name}_optuna_best_params.json", "w"), indent=2)
                print(f"  XGBoost_optuna F1-macro {rows[-1]['f1_macro']:.4f} | tuned {CONFIG['N_OPTUNA_TRIALS']} trials (val {bval:.4f})")
        except Exception as e:
            print(f"  [optuna] skipped: {type(e).__name__}: {str(e)[:70]}")
    if not rows:
        print(f"[{head_name}] no model trained"); return None
    lb = pd.DataFrame(rows).sort_values("f1_macro", ascending=False).reset_index(drop=True)
    lb.to_csv(out / f"{cat.lower()}_{head_name}_leaderboard.csv", index=False)
    champ = lb.iloc[0]["model"]; champ_f1 = float(lb.iloc[0]["f1_macro"])
    floor = CONFIG["SEVERITY_F1_FLOOR"] if head_name == "severity" else CONFIG["ROOTCAUSE_F1_FLOOR"]
    gate_pass = champ_f1 >= floor
    print(f"  champion = {champ} | macro-F1 {champ_f1:.4f} | gate {floor:.2f} -> {'PASS' if gate_pass else 'FAIL (review)'}")

    LBL = np.arange(n_cls); names0 = [str(x) for x in le.classes_]
    # target-class distribution (v3 'Failure_Level_Distribution')
    try:
        vc = pd.Series(ytr).map(dict(enumerate(names0))).value_counts()
        ax = vc[::-1].plot(kind="barh", figsize=(7, 4), color="#A9D18E")
        ax.set_title(f"PS3 {cat} {head_name} -- class distribution (train)"); plt.tight_layout()
        plt.savefig(out / f"{cat.lower()}_{head_name}_class_distribution.png", dpi=110); plt.show()
    except Exception: plt.close("all")
    # per-model confusion + ROC (matches v3's xgb_mc/lgb_mc/cb_mc set) -- champion-quality artifacts for all
    for mname, mdl in fitted.items():
        try:
            pp = proba_of[mname]; pd_ = pp.argmax(1)
            _plot_confusion(confusion_matrix(yte, pd_, labels=LBL), names0,
                            f"PS3 {cat} {head_name} -- {mname} confusion (test)",
                            out / f"{cat.lower()}_{head_name}_{mname}_confusion.png")
            _plot_roc_pr(yte, pp, names0, LBL, cat, head_name, out, tag=mname)
            fi = getattr(mdl, "feature_importances_", None)
            if fi is not None:
                s = pd.Series(fi, index=feats).sort_values(ascending=False).head(20)
                ax = s[::-1].plot(kind="barh", figsize=(6.5, 5.5), color="#F4B183")
                ax.set_title(f"PS3 {cat} {head_name} -- {mname} feature importance"); plt.tight_layout()
                plt.savefig(out / f"{cat.lower()}_{head_name}_{mname}_featimp.png", dpi=110); plt.show()
        except Exception: plt.close("all")

    # ---- exhaustive champion evaluation (labels pinned to LBL so shapes always align) ----
    model = fitted[champ]; proba = proba_of[champ]; pred = proba.argmax(axis=1)
    names = [str(x) for x in le.classes_]
    cm = confusion_matrix(yte, pred, labels=LBL)                     # FIX: always n_cls x n_cls
    _plot_confusion(cm, names, f"PS3 {cat} {head_name} -- confusion (test)",
                    out / f"{cat.lower()}_{head_name}_confusion.png")
    per_f1 = f1_score(yte, pred, labels=LBL, average=None, zero_division=0)
    per_prec = precision_score(yte, pred, labels=LBL, average=None, zero_division=0)
    per_rec = recall_score(yte, pred, labels=LBL, average=None, zero_division=0)
    support = np.array([(yte == k).sum() for k in LBL])
    _plot_perclass_f1(names, per_f1, f"PS3 {cat} {head_name} -- per-class F1",
                      out / f"{cat.lower()}_{head_name}_perclass_f1.png")
    _plot_roc_pr(yte, proba, names, LBL, cat, head_name, out)        # ROC + PR curves (OVR)
    _plot_calibration(yte, proba, pred, cat, head_name, out)         # reliability curve
    _write_report_txt(yte, pred, names, LBL, cat, head_name, champ, lb, floor, gate_pass, out)

    # extra rigorous metrics (beyond the v3 set)
    def _safe(fn, *a, **k):
        try: return round(float(fn(*a, **k)), 4)
        except Exception: return None
    bal_acc = _safe(balanced_accuracy_score, yte, pred)
    kappa = _safe(cohen_kappa_score, yte, pred)
    mcc = _safe(matthews_corrcoef, yte, pred)
    ll = _safe(log_loss, yte, proba, labels=LBL)

    # SHAP suite always renders on a supported tree model (proxy explainer if the champion is linear)
    _SHAP_OK = ("RandomForest", "XGBoost", "LightGBM", "CatBoost")
    shap_name = champ if champ in _SHAP_OK else next((m for m in lb["model"].tolist() if m in _SHAP_OK), champ)
    shap_imp = _shap_importance(fitted.get(shap_name, model), prep, Xte, feats, cat, head_name, out, names)

    summary = {"device_category": cat, "head": head_name, "target": target_col,
               "champion": champ, "classes": names, "n_classes": int(n_cls),
               "test_f1_macro": round(champ_f1, 4),
               "test_f1_weighted": round(float(lb.iloc[0]["f1_weighted"]), 4),
               "test_accuracy": round(float(lb.iloc[0]["accuracy"]), 4),
               "test_balanced_accuracy": bal_acc,
               "test_auc_macro_ovr": (round(float(lb.iloc[0]["auc_macro_ovr"]), 4)
                                      if not pd.isna(lb.iloc[0]["auc_macro_ovr"]) else None),
               "test_pr_auc_macro": (round(float(lb.iloc[0]["pr_auc_macro"]), 4)
                                     if not pd.isna(lb.iloc[0]["pr_auc_macro"]) else None),
               "test_cohen_kappa": kappa, "test_mcc": mcc, "test_log_loss": ll,
               "macro_f1_floor": floor, "gate_pass": bool(gate_pass),
               "n_train": int(len(tr)), "n_test": int(len(te)), "n_features": len(feats),
               "class_balance_train": {names[k]: int((ytr == k).sum()) for k in LBL},
               "per_class": {names[k]: {"precision": round(float(per_prec[k]), 4),
                                        "recall": round(float(per_rec[k]), 4),
                                        "f1": round(float(per_f1[k]), 4),
                                        "support": int(support[k])} for k in LBL}}
    json.dump(summary, open(out / f"{cat.lower()}_{head_name}_champion_summary.json", "w"), indent=2)

    # serving bundle
    import joblib
    joblib.dump({"prep": prep, "model": model, "label_encoder": le, "features": feats,
                 "head": head_name, "category": cat, "target": target_col},
                out / f"{cat.lower()}_{head_name}_bundle.joblib")

    return {"champ": champ, "model": model, "prep": prep, "le": le, "feats": feats,
            "summary": summary, "shap_imp": shap_imp, "gate_pass": gate_pass}

# ----------------------------------------------------------------------------- viz helpers
def _plot_confusion(cm, labels, title, path):
    fig, ax = plt.subplots(figsize=(1.6 + 0.7 * len(labels), 1.4 + 0.6 * len(labels)))
    cmn = cm.astype("float64") / cm.sum(axis=1, keepdims=True).clip(min=1)
    im = ax.imshow(cmn, cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(len(labels))); ax.set_yticks(range(len(labels)))
    ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=8); ax.set_yticklabels(labels, fontsize=8)
    for i in range(len(labels)):
        for j in range(len(labels)):
            ax.text(j, i, f"{cm[i,j]}", ha="center", va="center",
                    color="white" if cmn[i, j] > 0.5 else "#333", fontsize=7)
    ax.set_xlabel("predicted"); ax.set_ylabel("actual"); ax.set_title(title, fontsize=9)
    plt.tight_layout(); plt.savefig(path, dpi=110); plt.show()

def _plot_perclass_f1(labels, f1s, title, path):
    fig, ax = plt.subplots(figsize=(max(5, 0.7 * len(labels)), 4))
    ax.bar(range(len(labels)), f1s, color=PASTEL[:len(labels)] if len(labels) <= len(PASTEL) else None)
    ax.set_xticks(range(len(labels))); ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=8)
    ax.set_ylim(0, 1); ax.set_ylabel("F1"); ax.set_title(title, fontsize=10)
    for i, v in enumerate(f1s): ax.text(i, v + 0.02, f"{v:.2f}", ha="center", fontsize=8)
    plt.tight_layout(); plt.savefig(path, dpi=110); plt.show()

def _plot_roc_pr(y, proba, names, LBL, cat, head, out, tag=""):
    """One-vs-rest ROC + PR curves (test). Caps at 8 classes by support to stay legible."""
    Yoh = label_binarize(y, classes=LBL)
    if Yoh.shape[1] == 1: Yoh = np.hstack([1 - Yoh, Yoh])
    order = np.argsort([-(y == k).sum() for k in LBL])[:8]
    pre = f"{cat.lower()}_{head}{('_' + tag) if tag else ''}"
    # ROC
    fig, ax = plt.subplots(figsize=(6.5, 5.5))
    for k in order:
        try:
            fpr, tpr, _ = roc_curve(Yoh[:, k], proba[:, k]); a = roc_auc_score(Yoh[:, k], proba[:, k])
            ax.plot(fpr, tpr, lw=1.4, color=PASTEL[list(order).index(k) % len(PASTEL)],
                    label=f"{names[k][:22]} (AUC {a:.2f})")
        except Exception: pass
    ax.plot([0, 1], [0, 1], "--", color="#bbb", lw=1)
    ax.set_xlabel("FPR"); ax.set_ylabel("TPR"); ax.set_title(f"PS3 {cat} {head} -- ROC (OVR, test)", fontsize=10)
    ax.legend(fontsize=7, loc="lower right"); plt.tight_layout()
    plt.savefig(out / f"{pre}_roc_ovr.png", dpi=110); plt.show()
    # PR
    fig, ax = plt.subplots(figsize=(6.5, 5.5))
    for k in order:
        try:
            prec, rec, _ = precision_recall_curve(Yoh[:, k], proba[:, k])
            ap = average_precision_score(Yoh[:, k], proba[:, k])
            ax.plot(rec, prec, lw=1.4, color=PASTEL[list(order).index(k) % len(PASTEL)],
                    label=f"{names[k][:22]} (AP {ap:.2f})")
        except Exception: pass
    ax.set_xlabel("recall"); ax.set_ylabel("precision"); ax.set_title(f"PS3 {cat} {head} -- PR (OVR, test)", fontsize=10)
    ax.legend(fontsize=7, loc="lower left"); plt.tight_layout()
    plt.savefig(out / f"{pre}_pr_ovr.png", dpi=110); plt.show()

def _plot_calibration(y, proba, pred, cat, head, out):
    """Confidence reliability: bin max-proba vs empirical accuracy of the predicted class."""
    conf = proba.max(axis=1); correct = (pred == np.asarray(y)).astype(float)
    bins = np.linspace(0, 1, 11); idx = np.digitize(conf, bins) - 1
    xs, ys = [], []
    for b in range(10):
        m = idx == b
        if m.sum() >= 5: xs.append(conf[m].mean()); ys.append(correct[m].mean())
    fig, ax = plt.subplots(figsize=(5.5, 5))
    ax.plot([0, 1], [0, 1], "--", color="#bbb", lw=1, label="perfect")
    ax.plot(xs, ys, "o-", color="#9DC3E6", lw=1.6, label="model")
    ax.set_xlabel("mean predicted confidence"); ax.set_ylabel("empirical accuracy")
    ax.set_title(f"PS3 {cat} {head} -- reliability", fontsize=10); ax.legend(fontsize=8)
    plt.tight_layout(); plt.savefig(out / f"{cat.lower()}_{head}_calibration.png", dpi=110); plt.show()

def _write_report_txt(y, pred, names, LBL, cat, head, champ, lb, floor, gate_pass, out):
    rep = classification_report(y, pred, labels=LBL, target_names=names, zero_division=0, digits=4)
    with open(out / f"{cat.lower()}_{head}_report.txt", "w") as f:
        f.write(f"PS3 {cat} -- {head} head\n{'='*60}\n")
        f.write(f"champion         : {champ}\n")
        f.write(f"macro-F1 gate    : {floor:.2f} -> {'PASS' if gate_pass else 'FAIL (review)'}\n\n")
        f.write("model bake-off (sorted by macro-F1):\n")
        f.write(lb.to_string(index=False)); f.write("\n\n")
        f.write("per-class classification report (test):\n"); f.write(rep); f.write("\n")

def _norm_shap(sv, n_feat):
    """Normalise TreeExplainer output to a list of K arrays, each (n_samples x n_feat)."""
    if isinstance(sv, list):
        return sv
    if getattr(sv, "ndim", 2) == 3:
        if sv.shape[1] == n_feat:   return [sv[:, :, k] for k in range(sv.shape[2])]   # (n, n_feat, K)
        if sv.shape[2] == n_feat:   return [sv[k] for k in range(sv.shape[0])]         # (K, n, n_feat)
    return [np.asarray(sv)]

def _shap_importance(model, prep, X, feats, cat, head, out, class_names=None):
    """Global mean|SHAP| bar + beeswarm + per-class mean-SHAP + dependence (v3-style suite)."""
    pre = f"{cat.lower()}_{head}"
    try:
        if shap_mod is None: return None
        if not any(k in type(model).__name__ for k in ["Forest", "XGB", "LGBM", "Boosting", "CatBoost"]):
            return None
        Xt = pd.DataFrame(prep.transform(X), columns=feats)
        samp = Xt.sample(min(1500, len(Xt)), random_state=RS)
        expl = shap_mod.TreeExplainer(model)
        sv_list = _norm_shap(expl.shap_values(samp), len(feats))
        absmean = np.mean([np.abs(s) for s in sv_list], axis=0)           # classes-averaged |SHAP|
        imp = pd.Series(absmean.mean(axis=0), index=feats).sort_values(ascending=False)
        imp.to_csv(out / f"{pre}_shap_importance.csv")
        # (1) global bar
        ax = imp.head(20)[::-1].plot(kind="barh", figsize=(7, 6.5), color="#9DC3E6")
        ax.set_title(f"PS3 {cat} {head} -- global SHAP importance (classes averaged)"); plt.tight_layout()
        plt.savefig(out / f"{pre}_shap.png", dpi=110); plt.show()
        names = [str(c) for c in (class_names if class_names is not None else range(len(sv_list)))]
        focus = int(np.argmax([np.abs(s).mean() for s in sv_list]))       # most-influenced class
        # (2) global beeswarm (on the focus class -- signed)
        try:
            shap_mod.summary_plot(sv_list[focus], samp, max_display=20, show=False)
            plt.title(f"PS3 {cat} {head} -- SHAP beeswarm ({names[focus][:22]})", fontsize=9)
            plt.tight_layout(); plt.savefig(out / f"{pre}_shap_beeswarm.png", dpi=110, bbox_inches="tight"); plt.show()
        except Exception as e: print(f"  [shap beeswarm] {type(e).__name__}"); plt.close("all")
        # (3) per-class mean|SHAP| bars (top <=4 classes by influence)
        for k in np.argsort([-np.abs(s).mean() for s in sv_list])[:4]:
            try:
                cimp = pd.Series(np.abs(sv_list[k]).mean(0), index=feats).sort_values(ascending=False)
                ax = cimp.head(15)[::-1].plot(kind="barh", figsize=(6.5, 5.5), color="#C9A0DC")
                ax.set_title(f"PS3 {cat} {head} -- mean|SHAP| for class {names[k][:22]}"); plt.tight_layout()
                safe = names[k].replace(" ", "_").replace("/", "-")[:24]
                plt.savefig(out / f"{pre}_shap_class_{safe}.png", dpi=110); plt.show()
            except Exception: plt.close("all")
        # (4) dependence plots for the top-3 global features on the focus class
        for feat in imp.head(3).index:
            try:
                shap_mod.dependence_plot(feat, sv_list[focus], samp, interaction_index=None, show=False)
                plt.title(f"PS3 {cat} {head} -- SHAP dependence: {feat} ({names[focus][:16]})", fontsize=9)
                plt.tight_layout(); plt.savefig(out / f"{pre}_shap_dep_{feat}.png", dpi=110, bbox_inches="tight"); plt.show()
            except Exception: plt.close("all")
        print(f"  SHAP suite ({head}): global+beeswarm+per-class+dependence | top: {', '.join(imp.head(5).index)}")
        return imp
    except Exception as e:
        print(f"  [shap:{head}] skipped: {type(e).__name__}: {str(e)[:60]}"); plt.close("all"); return None

# ----------------------------------------------------------------------------- scoring -> device + serial + incident
def score_and_aggregate(df, cat, out, sev, rc):
    """Produce incident-grain, device-grain and serial-grain scored feeds (RDS-ready)."""
    inc = df.copy()
    # incident-grain predictions for each trained head
    def predict_head(h):
        if h is None: return None, None
        p = h["model"].predict_proba(h["prep"].transform(inc[h["feats"]].astype("float64")))
        lab = h["le"].inverse_transform(p.argmax(axis=1))
        conf = p.max(axis=1)
        return lab, conf
    sev_lab, sev_conf = predict_head(sev)
    rc_lab, rc_conf = predict_head(rc)

    base = ["availability_event_id", "device_id", "mars_device_category", "AE_START_DTM",
            "matched_serial_nbr", "component_age_days", "FACILITY_ID", "FACILITY_NAME"]
    inc_out = inc[[c for c in base if c in inc.columns]].copy()
    if sev is not None:
        inc_out["pred_severity"] = sev_lab
        inc_out["pred_severity_conf"] = np.round(sev_conf, 5)
        # Do NOT fillna("MAJOR"): MIN_CLASS_COUNT folds rare classes into "OTHER", which is in
        # no bucket. Defaulting unmapped codes to MAJOR is exactly what hid the label/code bug.
        _sev_codes = pd.Series(sev_lab).astype(str).str.strip().str.upper()
        _collapsed = _sev_codes.map(CONFIG["SEVERITY_COLLAPSE"])
        _unmapped = sorted(_sev_codes[_collapsed.isna()].unique().tolist())
        if _unmapped:
            print(f"[severity-collapse] WARNING unmapped codes -> UNKNOWN: {_unmapped}")
        inc_out["pred_severity_collapsed"] = _collapsed.fillna("UNKNOWN").values
        if CONFIG["SEVERITY_TARGET"] in inc.columns:
            inc_out["actual_severity"] = inc[CONFIG["SEVERITY_TARGET"]].astype(str).values
    if rc is not None:
        inc_out["pred_component"] = rc_lab
        inc_out["pred_component_conf"] = np.round(rc_conf, 5)
        if CONFIG["ROOTCAUSE_TARGET"] in inc.columns:
            inc_out["actual_component"] = inc[CONFIG["ROOTCAUSE_TARGET"]].astype(str).values
    inc_out.to_csv(out / f"{cat.lower()}_incident_predictions.csv", index=False)
    print(f"  incident predictions: {len(inc_out):,} rows -> {cat.lower()}_incident_predictions.csv")

    # device-grain rollup
    g = inc_out.groupby("device_id")
    dev = pd.DataFrame({"n_incidents": g.size()})
    if "pred_severity_collapsed" in inc_out:
        dev["pct_critical_pred"] = g["pred_severity_collapsed"].apply(lambda s: (s == "CRITICAL").mean()).round(4)
        dev["dominant_pred_severity"] = g["pred_severity"].agg(lambda s: s.value_counts().index[0])
    if "pred_component" in inc_out:
        dev["dominant_pred_component"] = g["pred_component"].agg(lambda s: s.value_counts().index[0])
    if "component_age_days" in inc_out:
        dev["avg_component_age_days"] = g["component_age_days"].mean().round(0)
    if "AE_START_DTM" in inc_out:
        dev["last_incident_dtm"] = g["AE_START_DTM"].max()
    dev = dev.reset_index().sort_values("n_incidents", ascending=False)
    dev["mars_device_category"] = cat
    dev.to_csv(out / f"{cat.lower()}_device_predictions.csv", index=False)
    print(f"  device rollup: {len(dev):,} devices -> {cat.lower()}_device_predictions.csv")

    # serial-grain rollup (the serial-level ask) keyed on matched_serial_nbr
    ser_out = None
    if "matched_serial_nbr" in inc_out.columns and inc_out["matched_serial_nbr"].notna().any():
        s = inc_out[inc_out["matched_serial_nbr"].notna()].copy()
        gs = s.groupby(["device_id", "matched_serial_nbr"])
        ser = pd.DataFrame({"n_incidents": gs.size()})
        if "component_age_days" in s: ser["component_age_days"] = gs["component_age_days"].max().round(0)
        if "pred_component" in s: ser["dominant_pred_component"] = gs["pred_component"].agg(lambda x: x.value_counts().index[0])
        if "pred_severity_collapsed" in s: ser["pct_critical_pred"] = gs["pred_severity_collapsed"].apply(lambda x: (x=="CRITICAL").mean()).round(4)
        if "AE_START_DTM" in s: ser["last_incident_dtm"] = gs["AE_START_DTM"].max()
        ser = ser.reset_index().sort_values("n_incidents", ascending=False)
        ser["mars_device_category"] = cat
        ser.to_csv(out / f"{cat.lower()}_serial_predictions.csv", index=False)
        ser_out = ser
        print(f"  serial rollup: {len(ser):,} serials -> {cat.lower()}_serial_predictions.csv")
    else:
        print("  [serial] no matched_serial_nbr populated for this device type")

    # explainability top-N per incident (from SHAP global as per-incident proxy is heavy; use global top-k)
    expl_rows = []
    for h, hd in [("severity", sev), ("root_cause", rc)]:
        if hd is None or hd.get("shap_imp") is None: continue
        for rank, (feat, val) in enumerate(hd["shap_imp"].head(5).items(), 1):
            expl_rows.append({"head": h, "feature_rank": rank, "feature_name": feat,
                              "shap_importance": round(float(val), 6)})
    if expl_rows:
        pd.DataFrame(expl_rows).to_csv(out / f"{cat.lower()}_prediction_explainability.csv", index=False)
    return inc_out, dev, ser_out

def per_type_viz(df, cat, out):
    # severity mix for top devices (stacked) + component-age vs incident-count scatter
    if CONFIG["SEVERITY_TARGET"] in df.columns:
        top_dev = df["device_id"].value_counts().head(12).index
        sub = df[df["device_id"].isin(top_dev)]
        mix = pd.crosstab(sub["device_id"], sub[CONFIG["SEVERITY_TARGET"]].astype(str))
        mix = mix.loc[top_dev]
        ax = mix.plot(kind="bar", stacked=True, figsize=(9, 5), color=PASTEL[:mix.shape[1]])
        ax.set_title(f"PS3 {cat} -- severity mix, top-12 incident devices"); ax.set_xlabel("")
        ax.legend(fontsize=7, ncol=2); plt.tight_layout()
        plt.savefig(out / f"{cat.lower()}_device_severity_mix.png", dpi=110); plt.show()
    if {"component_age_days", "device_id"}.issubset(df.columns):
        agg = df.groupby("device_id").agg(n=("availability_event_id", "size"),
                                          age=("component_age_days", "mean")).dropna()
        if len(agg):
            fig, ax = plt.subplots(figsize=(7, 5))
            ax.scatter(agg["age"], agg["n"], s=18, alpha=0.5, color="#F4B183")
            ax.set_xlabel("avg component age (days)"); ax.set_ylabel("incident count")
            ax.set_title(f"PS3 {cat} -- component age vs incident volume (device)")
            plt.tight_layout(); plt.savefig(out / f"{cat.lower()}_age_vs_incidents.png", dpi=110); plt.show()

# ----------------------------------------------------------------------------- device-type runner
def run_device_type(df_all, cat, out_root):
    out = Path(out_root) / CONFIG["SUBFOLDER"][cat]; out.mkdir(parents=True, exist_ok=True)
    d = df_all[df_all["mars_device_category"] == cat].copy()
    print("\n" + "=" * 74); print(f" DEVICE TYPE: {cat}   incidents = {len(d):,}"); print("=" * 74)
    if len(d) == 0:
        return handle_no_incidents(cat, out)
    d = add_temporal(add_facility_freq(d))
    d["_ae_date"] = pd.to_datetime(d["AE_START_DTM"], errors="coerce").dt.normalize()
    d = d[d["_ae_date"].notna()].sort_values("_ae_date").reset_index(drop=True)
    feats = build_feature_list(d)
    split_date = d["_ae_date"].quantile(0.80)

    scan, leaky = leakage_scan(d, CONFIG["SEVERITY_TARGET"], feats, split_date)
    feats = [c for c in feats if c not in set(leaky)]
    nzv = [c for c in feats if d[c].nunique(dropna=True) < CONFIG["NZV_UNIQUE_MIN"]]
    feats = [c for c in feats if c not in set(nzv)]
    if len(scan): scan.round(4).to_csv(out / f"{cat.lower()}_leakage_scan.csv")
    json.dump({"features": feats, "dropped_leaky": leaky, "dropped_nzv": nzv,
               "monitored": [m for m in CONFIG["MONITORED_FEATURES"] if m in feats]},
              open(out / f"{cat.lower()}_feature_manifest.json", "w"), indent=2)
    print(f"final feature count: {len(feats)} | dropped leaky {len(leaky)} nzv {len(nzv)}")

    if len(d) < CONFIG["MIN_INCIDENTS_TO_MODEL"]:
        print(f"[{cat}] only {len(d)} incidents (< {CONFIG['MIN_INCIDENTS_TO_MODEL']}) -> REPORT-ONLY (too sparse to model honestly)")
        _sparse_report(d, cat, out)
        return {"cat": cat, "modeled": False, "n_incidents": len(d)}

    sev = train_head(d, feats, CONFIG["SEVERITY_TARGET"], "severity", cat, out, split_date)
    rc = train_head(d, feats, CONFIG["ROOTCAUSE_TARGET"], "root_cause", cat, out, split_date)
    inc_out, dev, ser = score_and_aggregate(d, cat, out, sev, rc)
    per_type_viz(d, cat, out)
    return {"cat": cat, "modeled": True, "n_incidents": len(d),
            "severity": sev["summary"] if sev else None, "root_cause": rc["summary"] if rc else None,
            "n_devices": int(dev["device_id"].nunique()) if dev is not None else 0,
            "n_serials": int(len(ser)) if ser is not None else 0}

def _sparse_report(d, cat, out):
    rep = {"device_category": cat, "n_incidents": int(len(d)),
           "note": f"{cat} has too few PS3 incidents to train an honest classifier; descriptive only.",
           "severity_distribution": d[CONFIG["SEVERITY_TARGET"]].astype(str).value_counts().to_dict()
                                     if CONFIG["SEVERITY_TARGET"] in d else {},
           "component_distribution": d[CONFIG["ROOTCAUSE_TARGET"]].astype(str).value_counts().to_dict()
                                     if CONFIG["ROOTCAUSE_TARGET"] in d else {}}
    json.dump(rep, open(out / f"{cat.lower()}_sparse_report.json", "w"), indent=2)

def handle_no_incidents(cat, out):
    # VALIDATOR path: 0 availability-event incidents. Honest stub + OOS proxy from device_failures.
    print(f"[{cat}] 0 PS3 availability-event incidents (confirmed by data foundation). Writing honest stub + OOS proxy.")
    stub = {"device_category": cat, "n_ps3_incidents": 0,
            "why": ("PS3 is sourced from silver.incident_root_cause (ServiceNow availability events). "
                    f"{cat}/BMV failures are NOT recorded as availability events -- they surface as "
                    "device_event out-of-service 'Set' events (dim_event_matrix OOS flags). So the "
                    "PS3 availability-event feed has 0 rows for this device type by construction."),
            "recommendation": ("Score validators via the PS4 anomaly head (unsupervised, covers all 3 types) "
                               "and the OOS-proxy severity below; a true PS3 validator model needs Cubic to "
                               "emit validator availability events or fault codes."),
            "proxy_source": "silver.device_failures (S26) OOS 'Set' events, severity via dim_failure_level"}
    json.dump(stub, open(out / f"{cat.lower()}_ps3_stub.json", "w"), indent=2)
    # try the OOS proxy
    df_fail = load_table("mars_dev.silver.device_failures",
                         f"{CONFIG['SILVER_PREFIX']}/device_failures", category_filter=cat, required=False)
    if df_fail is not None and len(df_fail):
        sevcol = next((c for c in ["failure_level", "FAILURE_LEVEL", "failure_level_label"] if c in df_fail.columns), None)
        if sevcol:
            vc = df_fail[sevcol].astype(str).value_counts()
            vc.to_csv(out / f"{cat.lower()}_oos_proxy_severity.csv")
            ax = vc.head(10).plot(kind="bar", figsize=(7, 4), color="#C9A0DC")
            ax.set_title(f"PS3 {cat} -- OOS-proxy severity (device_failures, NOT availability events)")
            plt.tight_layout(); plt.savefig(out / f"{cat.lower()}_oos_proxy_severity.png", dpi=110); plt.show()
            print(f"  OOS proxy: {len(df_fail):,} {cat} OOS events, {df_fail[sevcol].nunique()} severity levels")
    else:
        print(f"  [proxy] device_failures export for {cat} not available; wrote stub only.")
    return {"cat": cat, "modeled": False, "n_incidents": 0, "stub": True}

## 7. TVM — the dominant device type (both heads)
Full two-head model. Watch the leakage scan (top solo-AUC should be `sn_event_code_id` ≈ 0.76 on real data — flagged, not dropped), the macro-F1 gate, the confusion matrices, SHAP drivers, and the device + serial rollups.

In [ ]:
res_tvm = run_device_type(df, "TVM", CONFIG["OUT_ROOT"])

## 8. GATE — sparse device type (honest caveat)
GATE has far fewer PS3 incidents than TVM. The engine still trains both heads but reports honestly; if incidents fall below `MIN_INCIDENTS_TO_MODEL` it drops to a descriptive report rather than pretending to model.

In [ ]:
res_gate = run_device_type(df, "GATE", CONFIG["OUT_ROOT"])

## 9. VALIDATOR — honest stub + OOS proxy
`gold.device_ps3_incident` has **0 VALIDATOR rows** (validator/BMV failures are device_event OOS `Set` events, not availability events). The engine writes an honest explainer to `validators/` and, where `silver.device_failures` is exported, an **OOS-proxy severity** view — clearly labelled as a proxy, not a PS3 availability-event model.

In [ ]:
res_val = run_device_type(df, "VALIDATOR", CONFIG["OUT_ROOT"])

## 10. Finalize — run summary, model-registry rows & RDS load manifest
Writes `ps3_run_summary.json`, `ps3_model_registry_rows.csv` (feeds `ml_model_registry` / `ml_models`), and `ps3_rds_load_manifest.json` (maps each CSV to its RDS target table for the daily loader).

In [ ]:
def _finalize(results, df):
    out = Path(CONFIG["OUT_ROOT"])
    summ = {"city_id": CONFIG["CITY_ID"], "n_incidents": int(len(df)),
            "device_scope": CONFIG["DEVICE_SCOPE"], "results": results}
    json.dump(summ, open(out / "ps3_run_summary.json", "w"), indent=2, default=str)
    reg = []
    for r in results:
        if not r or not r.get("modeled"): continue
        for head in ["severity", "root_cause"]:
            s = r.get(head)
            if s:
                reg.append({"ps_id": "PS3", "device_category": r["cat"], "prediction_head": head,
                            "model_name": f"ps3-{head}-{r['cat'].lower()}", "algorithm": s["champion"],
                            "test_f1_macro": s["test_f1_macro"], "test_auc_macro": s.get("test_auc_macro_ovr"),
                            "gate_pass": s["gate_pass"], "n_classes": s["n_classes"]})
    if reg: pd.DataFrame(reg).to_csv(out / "ps3_model_registry_rows.csv", index=False)
    load_manifest = {"rds_targets": {
        "ps3_incident_predictions": [f"{CONFIG['SUBFOLDER'][r['cat']]}/{r['cat'].lower()}_incident_predictions.csv"
                                     for r in results if r and r.get("modeled")],
        "ps3_device_predictions": [f"{CONFIG['SUBFOLDER'][r['cat']]}/{r['cat'].lower()}_device_predictions.csv"
                                   for r in results if r and r.get("modeled")],
        "ps3_serial_predictions": [f"{CONFIG['SUBFOLDER'][r['cat']]}/{r['cat'].lower()}_serial_predictions.csv"
                                   for r in results if r and r.get("modeled") and r.get("n_serials")],
    }}
    json.dump(load_manifest, open(out / "ps3_rds_load_manifest.json", "w"), indent=2)
    print("\n" + "=" * 74); print(" PS3 RUN COMPLETE"); print("=" * 74)
    for r in results:
        if not r: continue
        if r.get("modeled"):
            sv = r.get("severity"); rc = r.get("root_cause")
            print(f" {r['cat']:10s} incidents {r['n_incidents']:>6,} | "
                  f"severity F1 {sv['test_f1_macro'] if sv else 'NA'} ({'PASS' if sv and sv['gate_pass'] else 'review'}) | "
                  f"root-cause F1 {rc['test_f1_macro'] if rc else 'NA'} ({'PASS' if rc and rc['gate_pass'] else 'review'}) | "
                  f"devices {r.get('n_devices',0)} serials {r.get('n_serials',0)}")
        else:
            print(f" {r['cat']:10s} incidents {r.get('n_incidents',0):>6,} | REPORT-ONLY / stub ({'validator OOS proxy' if r.get('stub') else 'sparse'})")
    print(" artifacts under", out.resolve())

# patch loader so the synthetic validator proxy works offline

_finalize([res_tvm, res_gate, res_val], df)

## 11. Next steps — wire to RDS + dashboard (daily)

The CSV feeds this notebook writes map 1:1 onto the additive RDS migration shipped with this delivery
(`rds/04_phase1b_ps3_rootcause_and_serial.sql`):

| Notebook output | RDS table |
|---|---|
| `{cat}_incident_predictions.csv` | `ps3_incident_predictions` (both heads, one row per incident × head) |
| `{cat}_device_predictions.csv` | `ps3_device_predictions` (device grain) |
| `{cat}_serial_predictions.csv` | `ps3_serial_predictions` (serial grain — `matched_serial_nbr`) |
| `{cat}_*_shap_importance.csv` / `{cat}_prediction_explainability.csv` | `ps3_prediction_explainability` |
| `ps3_model_registry_rows.csv` | `ml_model_registry` / `ml_models` |

**Daily production path** (see `docs/PS3_End_to_End_Architecture_Daily.md`): Databricks incremental silver+gold (MERGE on new `availability_event_id`) → S3 export → SageMaker batch scoring (`inference.py`, both heads, `/ping`+`/invocations` tested locally first) → predictions to S3 → `ps3_rds_writer.py` upserts to RDS → the existing Lambda API + React dashboard read the new tables. Set `MLFLOW_TRACKING_URI` to log the champions to the SageMaker MLflow server and register to the `ps3-root-cause` / `ps3-severity` model package groups.